In [1]:
import pandas as pd

df = pd.read_csv("results/imputation_results.csv")

df["method"] = pd.Categorical(df["method"], categories=["Simple", "KNN", "MissForest"], ordered=True)
df["split"] = pd.Categorical(df["split"], categories=["train", "test"], ordered=True)
df["missing_rate"] = pd.Categorical(df["missing_rate"], categories=["10%", "25%"], ordered=True)

metrics = ["rmse", "mae", "r2"]

def mean_and_empirical95CI(x: pd.Series):
    return pd.Series({
        "mean": x.mean(),
        "ci_lower_95": x.quantile(0.025),
        "ci_upper_95": x.quantile(0.975),
    })

summary = (
    df.groupby(["split", "missing_rate", "method"], observed=True)[metrics]
      .apply(lambda g: g.apply(mean_and_empirical95CI))
      .unstack(-1) 
)

summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index().sort_values(["split", "missing_rate", "method"]).reset_index(drop=True)

summary

,split,missing_rate,method,rmse_mean,rmse_ci_lower_95,rmse_ci_upper_95,mae_mean,mae_ci_lower_95,mae_ci_upper_95,r2_mean,r2_ci_lower_95,r2_ci_upper_95
0,train,10%,Simple,0.931016,0.904723,0.962257,0.699610,0.668436,0.727327,-0.036450,-0.061172,-0.004084
1,train,10%,KNN,0.917196,0.894198,0.941977,0.696463,0.669591,0.718067,-0.005972,-0.034746,0.037074
2,train,10%,MissForest,0.910456,0.881877,0.939401,0.690949,0.656799,0.717055,0.008688,-0.035039,0.061746
3,train,25%,Simple,0.922100,0.904246,0.941032,0.693367,0.674771,0.712751,-0.003988,-0.036744,0.025932
4,train,25%,KNN,0.912767,0.892724,0.937476,0.693072,0.672862,0.711822,0.016235,-0.014681,0.047081
5,train,25%,MissForest,0.987127,0.948324,1.032724,0.748955,0.717778,0.786938,-0.151124,-0.260441,-0.057352
6,test,10%,Simple,0.931946,0.893484,0.997724,0.697859,0.660155,0.744742,-0.035540,-0.078846,0.001255
7,test,10%,KNN,0.918069,0.877774,0.982815,0.696061,0.665705,0.740659,-0.005132,-0.060023,0.044314
8,test,10%,MissForest,0.921065,0.877107,0.976234,0.698591,0.656896,0.735070,-0.012041,-0.081415,0.054526
9,test,25%,Simple,0.917943,0.893616,0.948620,0.689931,0.667440,0.723566,-0.003651,-0.030691,0.030744


In [1]:
# Classification results (imputed data)

import pandas as pd

df = pd.read_csv("results/combined.csv")
df = df[df["classifier"].isin(["LogisticRegression","XGBoost","NeuralNetwork"])]

for i in df["imputer"].unique():
    for j in sorted(df["missing_rate"].unique()):

        subset = df[
            (df["imputer"] == i) &
            (df["missing_rate"] == j)]

        if subset.empty:
            continue

        print(f"\n Imputation Method: {i} | Missing Rate: {j}")

        summary_table = (subset.groupby("classifier")[["accuracy", "f1", "roc_auc"]].mean().reset_index())
        print(summary_table)


 Imputation Method: simple | Missing Rate: 10%
           classifier  accuracy        f1   roc_auc
0  LogisticRegression  0.869882  0.869256  0.945216
1       NeuralNetwork  0.836078  0.835587  0.921479
2             XGBoost  0.820980  0.819870  0.907472

 Imputation Method: simple | Missing Rate: 25%
           classifier  accuracy        f1   roc_auc
0  LogisticRegression  0.811608  0.811132  0.898654
1       NeuralNetwork  0.772745  0.773262  0.859608
2             XGBoost  0.776392  0.775359  0.861227

 Imputation Method: knn | Missing Rate: 10%
           classifier  accuracy        f1   roc_auc
0  LogisticRegression  0.870275  0.869534  0.944349
1       NeuralNetwork  0.837765  0.836892  0.921993
2             XGBoost  0.819922  0.818811  0.904447

 Imputation Method: knn | Missing Rate: 25%
           classifier  accuracy        f1   roc_auc
0  LogisticRegression  0.805529  0.804704  0.894579
1       NeuralNetwork  0.768627  0.768421  0.853507
2             XGBoost  0.766784  0

In [ ]:
# Classification results (complete data)

import pandas as pd

df = pd.read_csv("results/classification_results_complete.csv")

df_filtered = df[df["classifier"].isin(["LogisticRegression","XGBoost","NeuralNetwork"])]

print(f"\n Complete")

summary_table = (
    df_filtered
    .groupby("classifier")[["accuracy", "f1", "roc_auc"]]
    .mean()
    .reset_index()
)

print(summary_table)